In [2]:
# Zurich
from training_utilities_2nd_part import *

from variables_to_specify_zurich import *
df, columns_to_normalize, zurich_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, zurich_drop_columnss, zurich_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_zurich()

print(zurich_target_col)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

df = df.dropna().reset_index(drop=True)
zurich_df = df
zurich_time_steps = 1
zurich_df['Timestamp'] = pd.to_datetime(zurich_df['Timestamp'])
zurich_df= zurich_df[zurich_df['Timestamp'].dt.year == 2020]

Value_NE5


# stationary

In [5]:
zurich_len_of_training_data_of_stationary_model =14*No_of_datapoints_in_one_day

stationary_model = new_copied_lstm_statinary(zurich_df, zurich_len_of_training_data_of_stationary_model, zurich_target_col, zurich_drop_columnss, zurich_time_steps)

Epoch 1/10


2025-04-10 19:17:35.261416: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0516
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0233
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0157
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0124
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0107
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0098
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0091
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0088
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0085
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0084
X_train shape: (1344, 1, 9)
y_train shape: (1344,)
X_train mean: 0.361658
X_train std: 0.31298572
y_train mean: 0.31781477
y_train std: 0.19624063
y_train min: 0.025223356
y_train max: 0.70297104
total_test_error is : 0.013628208
total_test_error_mae is:  0.09099982
total_time is:  9.369307709
train time is :  0
Model Type: Sequential
Storage Required: 0.0

# Model reuse

In [4]:
# Model reuse
daily_df_avg = get_elect_daily_avg(zurich_df, No_of_datapoints_in_one_day, zurich_target_col, avg_target_col_name)

seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 14)

Detected seasonality periods (ACF): [  7  14  21  28  35  42  49  56  63  70  77  84  91  98 105 112 119 126
 133 140 147 154 161 168 175 182]
median_value is:  94


In [7]:
ratio_wass = len(filtered_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_tvd = len(filtered_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]
ratio_forecasted_wass = len(filtered_forecasted_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_forecasted_tvd = len(filtered_forecasted_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]

print('ratio_wass:', ratio_wass)
print('ratio_tvd:', ratio_tvd)
print('ratio_forecasted_wass:', ratio_forecasted_wass)
print('ratio_forecasted_tvd:', ratio_forecasted_tvd)

ratio_wass: 0.46153846153846156
ratio_tvd: 0.6923076923076923
ratio_forecasted_wass: 0.5
ratio_forecasted_tvd: 0.6153846153846154


## drift detection

In [8]:
df_copy = zurich_df[[zurich_target_col]]
target_col = zurich_target_col
time_steps = zurich_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
window_length=[14]
multiplier = No_of_datapoints_in_one_day

x = 14* multiplier
window_len_=[x]

window_length = [x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=zurich_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insuff

In [8]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage1= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model, zurich_len_of_training_data_of_stationary_model, zurich_df, "SA", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_wass))


window is:  1344
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0487
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0229
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0150
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0118
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0102
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0093
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0086
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0083
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0081
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0079
i/window is :  1.0
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0732
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0291
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0189
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0126
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0

In [9]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage2= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model, zurich_len_of_training_data_of_stationary_model, zurich_df, "SA", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_tvd))


window is:  1344
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0487
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0229
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0150
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0118
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0102
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0093
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0086
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0083
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0081
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0079
i/window is :  1.0
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0732
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0291
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0189
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0126
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0

In [10]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage3= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model, zurich_len_of_training_data_of_stationary_model, zurich_df, "ES", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_wass))


window is:  1344
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0487
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0229
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0150
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0118
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0102
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0093
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0086
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0083
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0081
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0079
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0732
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0291
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0189
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0126
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss:

In [11]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage4= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model, zurich_len_of_training_data_of_stationary_model, zurich_df, "ES", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_tvd))


window is:  1344
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0487
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0229
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0150
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0118
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0102
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0093
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0086
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0083
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0081
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0079
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0732
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0291
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0189
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0126
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss:

In [12]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print("avg_ml_storage_reuse is : ", avg_ml_storage_reuse)


avg_ml_storage_reuse is :  0.06299209594726562


# informed retraining

In [13]:
lstm_informed_update(stationary_model,zurich_df, zurich_target_col, zurich_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices)

window is:  1344
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0487
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0229
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0150
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0118
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0102
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0093
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0086
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0083
Epoch 9/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0081
Epoch 10/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0079
Model Type: Sequential
Storage Required: 0.06 MB
window is:  2688
Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0918
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0325
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0232
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss:

# periodical

In [9]:
mean_mse_per_window, min_index, mean_mae_per_window = periodical_lstm_training(zurich_df, zurich_target_col, zurich_time_steps, zurich_windows, zurich_drop_columnss)

window size is :  480
Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0173
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0088
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0079
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0071
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0065
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0062
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0058
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0054
Epoch 9/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0051
Epoch 10/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0048
Model Type: Sequential
Storage Required: 0.06 MB
Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.1658
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0423
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0362
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0317
Epoc